# Week 4 - PyTorch Training Structure and Hyperparameters

This notebook turns the neural-network ideas from the previous weeks into the standard PyTorch workflow used in many computer vision projects. The goal is not just to make a model run; the goal is to see where each responsibility belongs.

A typical PyTorch training notebook has these parts:

1. Select a computation device, such as GPU or CPU.
2. Build a `Dataset` that returns one example at a time.
3. Build `DataLoader`s that batch, shuffle, and parallelize data loading.
4. Define an `nn.Module` model.
5. Choose the loss function, optimizer, scheduler, and other hyperparameters.
6. Run a training loop with a separate validation phase.
7. Save the best model checkpoint.

The code below keeps those pieces separate so it is easier to debug and easier to modify later.


In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as transforms
from PIL import Image
from pathlib import Path

# Reproducibility: this makes many random choices repeatable.
# Some GPU operations can still be nondeterministic, but this is a good default.
torch.manual_seed(0)

# Device selection controls where tensors and model parameters live.
# CUDA is NVIDIA GPU support. MPS is Apple Silicon GPU support.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: cuda


## Dataset

A PyTorch `Dataset` represents the raw examples. Its job is to answer two questions:

- How many examples are available?
- For a particular index, what image and label should be returned?

The dataset should not train the model, update weights, or create batches. It should only load one example at a time. This version expects the common image-classification folder layout:

```text
data/train/class_0/example.jpg
data/train/class_1/example.jpg
data/val/class_0/example.jpg
data/val/class_1/example.jpg
```

Each subfolder name becomes a class name. The class names are converted into integer labels because `CrossEntropyLoss` expects class labels such as `0`, `1`, `2`, and so on.


In [2]:
class ImageClassificationDataset(Dataset):
    """Image classification dataset using one folder per class."""

    def __init__(self, root_directory, image_transform=None, class_to_index=None):
        self.root_directory = Path(root_directory)
        self.image_transform = image_transform

        # Accept the most common image file extensions.
        self.image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        # Infer class names from subdirectories, unless a mapping was provided.
        # Passing the training mapping into validation keeps labels consistent.
        if class_to_index is None:
            class_names = sorted(
                path.name
                for path in self.root_directory.iterdir()
                if path.is_dir()
            )
            self.class_to_index = {
                class_name: class_index
                for class_index, class_name in enumerate(class_names)
            }
        else:
            self.class_to_index = dict(class_to_index)

        self.samples = []

        # Store pairs of (image_path, integer_label).
        # __getitem__ will use these pairs later to load one example at a time.
        for class_name, class_index in self.class_to_index.items():
            class_directory = self.root_directory / class_name
            if not class_directory.exists():
                continue

            for image_path in sorted(class_directory.rglob("*")):
                if image_path.suffix.lower() in self.image_extensions:
                    self.samples.append((image_path, class_index))

        if len(self.samples) == 0:
            raise ValueError(
                f"No images found in {self.root_directory}. "
                "Expected folders like root/class_name/image.jpg."
            )

    def __len__(self):
        # PyTorch calls this to know how many examples are in the dataset.
        return len(self.samples)

    def __getitem__(self, sample_index):
        # PyTorch calls this to retrieve one example.
        image_path, class_label = self.samples[sample_index]

        # Convert every image to RGB so all examples have 3 channels.
        image = Image.open(image_path).convert("RGB")

        # Transforms handle resizing, augmentation, tensor conversion, and normalization.
        if self.image_transform is not None:
            image = self.image_transform(image)

        return image, class_label


## Transforms

Transforms define what happens to an image before it reaches the model. Training transforms often include randomness because random crops, flips, and color changes act as data augmentation. Augmentation makes the model see slightly different versions of the training images, which can reduce overfitting.

Validation transforms should usually be deterministic. We want validation accuracy to measure the model, not the luck of a random crop.

The normalization constants below are ImageNet defaults. They are especially useful when using ImageNet-pretrained models, but they are also reasonable defaults for many RGB image experiments.


In [3]:
training_transforms = transforms.Compose([
    # RandomResizedCrop changes both image scale and viewpoint.
    # This is a hyperparameter: size and scale control how aggressive the crop is.
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.7, 1.0),
    ),

    # Only use flipping when it does not change the meaning of the label.
    transforms.RandomHorizontalFlip(p=0.5),

    # Color jitter helps the model rely less on exact lighting or saturation.
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
    ),

    # Convert PIL images to PyTorch image tensors.
    transforms.ToImage(),

    # Convert integer pixel values to floating point values in [0, 1].
    transforms.ToDtype(dtype=torch.float32, scale=True),

    # Normalize each color channel.
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

validation_transforms = transforms.Compose([
    # Validation preprocessing is fixed so metrics are comparable across epochs.
    transforms.Resize(size=256),
    transforms.CenterCrop(size=224),
    transforms.ToImage(),
    transforms.ToDtype(dtype=torch.float32, scale=True),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


## Datasets and DataLoaders

After defining the dataset class and transforms, we create separate datasets for training and validation. The training dataset uses random augmentation; the validation dataset uses fixed preprocessing.

A `DataLoader` wraps a dataset and handles batching. Important DataLoader hyperparameters include:

- `batch_size`: number of examples per gradient update.
- `shuffle`: usually `True` for training and `False` for validation.
- `num_workers`: number of CPU worker processes used to load data.
- `pin_memory`: can speed up CPU-to-GPU transfer when using CUDA.

Update the paths below to match the dataset location on your machine.


In [4]:
training_dataset = ImageClassificationDataset(
    root_directory="../../datasets/imagenette2/train",
    image_transform=training_transforms,
)

validation_dataset = ImageClassificationDataset(
    root_directory="../../datasets/imagenette2/val",
    image_transform=validation_transforms,
    class_to_index=training_dataset.class_to_index,
)

# The number of classes is inferred from the training folders.
number_of_classes = len(training_dataset.class_to_index)
print(f"Number of classes: {number_of_classes}")
print(f"Training examples: {len(training_dataset)}")
print(f"Validation examples: {len(validation_dataset)}")

# pin_memory is useful for CUDA, but not needed for CPU or Apple MPS.
pin_memory = device.type == "cuda"

training_dataloader = DataLoader(
    dataset=training_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=pin_memory,
)

validation_dataloader = DataLoader(
    dataset=validation_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=pin_memory,
)


Number of classes: 10
Training examples: 9469
Validation examples: 3925


## Model

The model below is a multilayer perceptron (MLP). It is intentionally simple: it flattens each image into one long vector and then applies fully connected layers.

This is not the strongest architecture for image classification because flattening discards spatial structure. Convolutional neural networks and vision transformers usually work better for images. However, an MLP is a useful teaching model because the flow is easy to see:

```text
image tensor -> flattened vector -> hidden layers -> class logits
```

The final layer outputs logits, not probabilities. Logits are raw class scores. We pass logits directly into `CrossEntropyLoss`; we do not apply softmax first.


In [5]:
class ImageClassificationMultilayerPerceptron(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        image_channels = 3
        image_height = 224
        image_width = 224
        number_of_input_features = image_channels * image_height * image_width

        self.network = nn.Sequential(
            # [batch_size, 3, 224, 224] -> [batch_size, 150528]
            nn.Flatten(),

            nn.Linear(
                in_features=number_of_input_features,
                out_features=512,
            ),
            nn.ReLU(),

            # Dropout randomly turns off hidden activations during training.
            # This is a regularization hyperparameter.
            nn.Dropout(p=0.2),

            nn.Linear(
                in_features=512,
                out_features=256,
            ),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            # One output logit per class.
            nn.Linear(
                in_features=256,
                out_features=number_of_classes,
            ),
        )

    def forward(self, input_images):
        return self.network(input_images)


classification_model = ImageClassificationMultilayerPerceptron(
    number_of_classes=number_of_classes,
).to(device)

print(classification_model)


ImageClassificationMultilayerPerceptron(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=150528, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=256, out_features=10, bias=True)
  )
)


## Loss Function, Optimizer, and Scheduler

These choices are hyperparameters because they shape how learning happens.

`CrossEntropyLoss` is the standard loss for multiclass classification. It expects logits with shape `[batch_size, number_of_classes]` and integer labels with shape `[batch_size]`. Label smoothing makes the target labels slightly less extreme, which can improve generalization.

`AdamW` is a strong default optimizer. It combines adaptive learning rates with decoupled weight decay. The learning rate controls step size, while weight decay discourages very large weights.

The scheduler changes the learning rate over time. Cosine annealing gradually lowers the learning rate during training, allowing larger updates early and smaller refinements later.


In [6]:
loss_function = nn.CrossEntropyLoss(
    label_smoothing=0.05,
)

optimizer = torch.optim.AdamW(
    params=classification_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

number_of_epochs = 50

learning_rate_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer,
    T_max=number_of_epochs,
)

# Keep a history dictionary so we can plot learning curves later.
history = {
    "training_loss": [],
    "validation_loss": [],
    "validation_accuracy": [],
}


## Training and Validation Loop

Each epoch has two phases.

During training, the model uses random augmentation, computes gradients, and updates weights. During validation, the model uses deterministic preprocessing, does not compute gradients, and does not update weights.

The most important PyTorch loop commands are:

- `model.train()` turns on training behavior such as dropout.
- `optimizer.zero_grad()` clears old gradients.
- `loss.backward()` computes new gradients using backpropagation.
- `optimizer.step()` updates the weights.
- `model.eval()` turns off training-only behavior.
- `torch.no_grad()` avoids storing gradient information during validation.

The loop also saves the model whenever validation accuracy improves. This is called best-checkpoint saving.


In [ ]:
best_validation_accuracy = 0.0

for epoch_index in range(number_of_epochs):
    # --------------------------------------------------------
    # Training phase
    # --------------------------------------------------------
    classification_model.train()
    total_training_loss = 0.0

    for training_images, training_labels in training_dataloader:
        # Move the batch to the same device as the model.
        training_images = training_images.to(device, non_blocking=True)
        training_labels = training_labels.to(device, non_blocking=True)

        # Forward pass: compute predicted logits.
        predicted_training_logits = classification_model(training_images)

        # Compare predictions with the true labels.
        training_loss = loss_function(
            predicted_training_logits,
            training_labels,
        )

        # Backpropagation follows the standard PyTorch order:
        # clear old gradients -> compute new gradients -> update parameters.
        optimizer.zero_grad(set_to_none=True)
        training_loss.backward()

        # Gradient clipping prevents extremely large gradients from causing
        # unstable parameter updates.
        torch.nn.utils.clip_grad_norm_(
            parameters=classification_model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        # Multiply by batch size so the epoch average is weighted correctly.
        total_training_loss += training_loss.item() * training_images.size(0)

    learning_rate_scheduler.step()
    average_training_loss = total_training_loss / len(training_dataset)

    # --------------------------------------------------------
    # Validation phase
    # --------------------------------------------------------
    classification_model.eval()
    total_validation_loss = 0.0
    number_of_correct_validation_predictions = 0
    number_of_validation_examples = 0

    with torch.no_grad():
        for validation_images, validation_labels in validation_dataloader:
            validation_images = validation_images.to(device, non_blocking=True)
            validation_labels = validation_labels.to(device, non_blocking=True)

            predicted_validation_logits = classification_model(validation_images)

            validation_loss = loss_function(
                predicted_validation_logits,
                validation_labels,
            )

            total_validation_loss += validation_loss.item() * validation_images.size(0)

            predicted_validation_classes = predicted_validation_logits.argmax(dim=1)
            number_of_correct_validation_predictions += (
                predicted_validation_classes == validation_labels
            ).sum().item()
            number_of_validation_examples += validation_labels.size(0)

    average_validation_loss = total_validation_loss / len(validation_dataset)
    validation_accuracy = (
        number_of_correct_validation_predictions
        / number_of_validation_examples
    )

    history["training_loss"].append(average_training_loss)
    history["validation_loss"].append(average_validation_loss)
    history["validation_accuracy"].append(validation_accuracy)

    # Save the model whenever validation accuracy improves.
    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(
            classification_model.state_dict(),
            "best_multilayer_perceptron.pt",
        )

    current_learning_rate = learning_rate_scheduler.get_last_lr()[0]
    print(
        f"Epoch {epoch_index + 1:03d} | "
        f"Training Loss: {average_training_loss:.4f} | "
        f"Validation Loss: {average_validation_loss:.4f} | "
        f"Validation Accuracy: {validation_accuracy:.4f} | "
        f"LR: {current_learning_rate:.6f}"
    )

print(f"Best Validation Accuracy: {best_validation_accuracy:.4f}")


Epoch 001 | Training Loss: 3.8168 | Validation Loss: 2.2703 | Validation Accuracy: 0.1896 | LR: 0.000300
Epoch 002 | Training Loss: 2.2605 | Validation Loss: 2.1698 | Validation Accuracy: 0.2257 | LR: 0.000299
Epoch 003 | Training Loss: 2.1626 | Validation Loss: 2.0659 | Validation Accuracy: 0.2966 | LR: 0.000297
Epoch 004 | Training Loss: 2.0785 | Validation Loss: 2.0351 | Validation Accuracy: 0.3065 | LR: 0.000295
Epoch 005 | Training Loss: 2.0551 | Validation Loss: 2.0079 | Validation Accuracy: 0.3259 | LR: 0.000293
Epoch 006 | Training Loss: 2.0183 | Validation Loss: 1.9767 | Validation Accuracy: 0.3409 | LR: 0.000289
Epoch 007 | Training Loss: 2.0031 | Validation Loss: 1.9545 | Validation Accuracy: 0.3529 | LR: 0.000286
Epoch 008 | Training Loss: 1.9791 | Validation Loss: 1.9377 | Validation Accuracy: 0.3656 | LR: 0.000281
Epoch 009 | Training Loss: 1.9519 | Validation Loss: 1.8981 | Validation Accuracy: 0.3732 | LR: 0.000277
Epoch 010 | Training Loss: 1.9546 | Validation Loss: 1.

## Validation Classification Report and Confusion Matrix

Accuracy gives one overall number, but it does not show which classes the model is confusing. A classification report breaks performance down by class using precision, recall, and F1-score.

A confusion matrix gives a visual summary of mistakes. Rows are the true labels, and columns are the predicted labels. A strong classifier should have most of its counts on the diagonal.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

# Load the best validation checkpoint saved during training.
# This evaluates the best model, not necessarily the model from the final epoch.
classification_model.load_state_dict(
    torch.load("best_multilayer_perceptron.pt", map_location=device)
)
classification_model.eval()

validation_predictions = []
validation_true_labels = []

with torch.no_grad():
    for validation_images, validation_labels in validation_dataloader:
        validation_images = validation_images.to(device, non_blocking=True)

        predicted_validation_logits = classification_model(validation_images)
        predicted_validation_classes = predicted_validation_logits.argmax(dim=1)

        # Move predictions and labels back to the CPU so scikit-learn can use them.
        validation_predictions.extend(predicted_validation_classes.cpu().tolist())
        validation_true_labels.extend(validation_labels.cpu().tolist())

# Convert the class mapping into a list ordered by numeric class index.
index_to_class = {
    class_index: class_name
    for class_name, class_index in training_dataset.class_to_index.items()
}
class_names = [
    index_to_class[class_index]
    for class_index in range(number_of_classes)
]
class_indices = list(range(number_of_classes))

print("Validation Classification Report")
print(
    classification_report(
        validation_true_labels,
        validation_predictions,
        labels=class_indices,
        target_names=class_names,
        zero_division=0,
    )
)

validation_confusion_matrix = confusion_matrix(
    validation_true_labels,
    validation_predictions,
    labels=class_indices,
)

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(
    confusion_matrix=validation_confusion_matrix,
    display_labels=class_names,
).plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d",
)

ax.set_title("Validation Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## What to Tune First

When a model trains poorly, tune only a few things at a time. Good first experiments are:

- Learning rate: often the most important optimizer hyperparameter.
- Batch size: affects memory use, gradient noise, and speed.
- Number of epochs: too few underfits, too many may overfit.
- Dropout probability: larger values regularize more strongly.
- Weight decay: discourages large weights and can reduce overfitting.
- Data augmentation strength: too little may overfit, too much may make training too hard.

A useful habit is to watch both training loss and validation loss. If training loss improves but validation loss gets worse, the model is probably overfitting. If both losses stay high, the model may be underfitting, learning too slowly, or receiving incorrectly prepared data.
